<a href="https://colab.research.google.com/github/pranta2003/Fetal-health-multimodal-prediction/blob/master/Capstone_fetalDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 — NOW verify your files, using the full path as data
import os

data_path = "/content/drive/MyDrive/fetal-health-project/data/raw"

expected_files = []
for yr in range(2016, 2024):
    expected_files.append(f"{yr}_NATAL.csv")
    expected_files.append(f"{yr}_FETAL_COD.csv")

print(f"Checking {data_path}\n")
missing = []
for fname in expected_files:
    full_path = os.path.join(data_path, fname)
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"gotcha {fname} — {size_mb:.1f} MB")
    else:
        print(f"not find MISSING: {fname}")
        missing.append(fname)

print(f"\n{'All 16 files found!' if not missing else f'{len(missing)} files still missing.'}")

Checking /content/drive/MyDrive/fetal-health-project/data/raw

gotcha 2016_NATAL.csv — 5972.4 MB
gotcha 2016_FETAL_COD.csv — 15.6 MB
gotcha 2017_NATAL.csv — 5945.1 MB
gotcha 2017_FETAL_COD.csv — 14.9 MB
gotcha 2018_NATAL.csv — 5855.1 MB
gotcha 2018_FETAL_COD.csv — 13.7 MB
gotcha 2019_NATAL.csv — 5787.4 MB
gotcha 2019_FETAL_COD.csv — 13.4 MB
gotcha 2020_NATAL.csv — 5575.2 MB
gotcha 2020_FETAL_COD.csv — 12.2 MB
gotcha 2021_NATAL.csv — 5652.4 MB
gotcha 2021_FETAL_COD.csv — 12.5 MB
gotcha 2022_NATAL.csv — 5661.8 MB
gotcha 2022_FETAL_COD.csv — 11.9 MB
gotcha 2023_NATAL.csv — 5552.5 MB
gotcha 2023_FETAL_COD.csv — 11.7 MB

All 16 files found!


In [ ]:
import pandas as pd
import os

# Session crashed, so restart your runtime first (Runtime > Restart session),
# then re-mount Drive before running this cell

os.makedirs("/content/drive/MyDrive/fetal-health-project/data/processed", exist_ok=True)

years = range(2016, 2024)

WANTED_COLS = ['ATTEND', 'BFACIL', 'BFACIL3', 'BMI', 'BMI_R', 'BWTR4', 'CIG_0', 'CIG_1',
    'CIG_2', 'CIG_3', 'CIG_REC', 'COMBGEST', 'DBWT', 'DLMP_MM', 'DLMP_YY', 'DMETH_REC',
    'DPLURAL', 'FAGECOMB', 'FAGEREC11', 'FAGERPT_FLG', 'F_CIGS_0', 'F_CIGS_1', 'F_CIGS_2',
    'F_CIGS_3', 'F_MEDUC', 'F_MM_AICU', 'F_MPCB', 'F_M_HT', 'F_PWGT', 'F_RF_CESAR',
    'F_RF_GDIAB', 'F_RF_GHYPER', 'F_RF_NCESAR', 'F_RF_PDIAB', 'F_RF_PHYPER', 'F_TOBACO',
    'F_WIC', 'ILLB_R', 'ILLB_R11', 'IMP_PLUR', 'IMP_SEX', 'LBO_REC', 'MAGER', 'MAGER14',
    'MAGER9', 'MAGE_IMPFLG', 'MAGE_REPFLG', 'MBSTATE_REC', 'MEDUC', 'ME_PRES', 'ME_ROUT',
    'ME_TRIAL', 'MHISPX', 'MM_AICU', 'MM_RUPT', 'MRACE15', 'MRACE6', 'MRACEHISP',
    'MRACEIMP', 'M_Ht_In', 'OBGEST_FLG', 'OEGest_Comb', 'PRECARE', 'PRIORDEAD',
    'PRIORLIVE', 'PWgt_R', 'RDMETH_REC', 'RESTATUS', 'RF_ARTEC', 'RF_CESAR', 'RF_CESARN',
    'RF_EHYPE', 'RF_FEDRG', 'RF_GDIAB', 'RF_GHYPE', 'RF_INFTR', 'SEX', 'WIC']

CHUNK_SIZE = 200_000
SAMPLE_PER_YEAR = 45_000

def get_available_cols(path, wanted_cols):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    available = [c for c in wanted_cols if c in header]
    missing = [c for c in wanted_cols if c not in header]
    return available, missing

def count_rows_chunked(path, one_col, chunksize=CHUNK_SIZE):
    """Cheap pass — reads only 1 column to count total rows without heavy memory use"""
    total = 0
    for chunk in pd.read_csv(path, usecols=[one_col], chunksize=chunksize, low_memory=False):
        total += len(chunk)
    return total

def sample_csv_chunked(path, available_cols, missing_cols, target_n, chunksize=CHUNK_SIZE):
    """Reads the file in chunks, keeping only a small random slice of each chunk —
    the full file is NEVER held in memory at once, which is what fixes the crash"""
    total_rows = count_rows_chunked(path, available_cols[0], chunksize)
    frac = min(1.0, target_n / total_rows) if total_rows > 0 else 0

    sampled_chunks = []
    for chunk in pd.read_csv(path, usecols=available_cols, chunksize=chunksize, low_memory=False):
        if frac < 1.0:
            chunk = chunk.sample(frac=frac, random_state=42)
        sampled_chunks.append(chunk)

    result = pd.concat(sampled_chunks, ignore_index=True)
    if len(result) > target_n:
        result = result.sample(n=target_n, random_state=42)
    for col in missing_cols:
        result[col] = pd.NA
    return result, total_rows

birth_frames = []
for yr in years:
    path = f"/content/drive/MyDrive/fetal-health-project/data/raw/{yr}_NATAL.csv"
    available, missing = get_available_cols(path, WANTED_COLS)
    if missing:
        print(f"{yr}: missing columns filled as NaN → {missing}")
    df_sample, total_rows = sample_csv_chunked(path, available, missing, SAMPLE_PER_YEAR)
    df_sample["OUTCOME"] = 0
    df_sample["YEAR"] = yr
    birth_frames.append(df_sample)
    print(f"{yr}: sampled {len(df_sample)} rows out of {total_rows:,} total")
    del df_sample

livebirth_data = pd.concat(birth_frames, ignore_index=True)
print(f"\nTotal live birth rows: {len(livebirth_data)}")
livebirth_data.to_csv("/content/drive/MyDrive/fetal-health-project/data/processed/livebirth_sampled_all_years.csv", index=False)
print("✅ Saved successfully")

2016: missing columns filled as NaN → ['F_RF_CESAR', 'F_RF_NCESAR', 'MHISPX', 'RF_CESAR', 'RF_CESARN']
2016: sampled 45000 rows out of 3,956,112 total
2017: missing columns filled as NaN → ['MHISPX']
2017: sampled 45000 rows out of 3,864,754 total
2018: sampled 44991 rows out of 3,801,534 total
2019: sampled 44997 rows out of 3,757,582 total
2020: sampled 44994 rows out of 3,619,826 total
2021: sampled 44993 rows out of 3,669,928 total
2022: sampled 44995 rows out of 3,676,029 total
2023: sampled 44991 rows out of 3,605,081 total

Total live birth rows: 359961
✅ Saved successfully


In [ ]:
stillbirth = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/stillbirth_all_years.csv")
livebirth = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/livebirth_sampled_all_years.csv")

final = pd.concat([stillbirth, livebirth], ignore_index=True)
final = final.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final shape:", final.shape)
print(final["OUTCOME"].value_counts())
print("\nMissing values per column (top 10):")
print(final.isna().sum().sort_values(ascending=False).head(10))

final.to_csv("/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv", index=False)
print("\n Saved clinical_training_data.csv — ready for train_clinical_model.py")

/tmp/ipykernel_738/1881883685.py:2: DtypeWarning: Columns (76) have mixed types. Specify dtype option on import or set low_memory=False.
  livebirth = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/livebirth_sampled_all_years.csv")


Final shape: (718136, 80)
OUTCOME
0    359961
1    358175
Name: count, dtype: int64

Missing values per column (top 10):
MAGE_IMPFLG    356422
FAGERPT_FLG    356102
IMP_PLUR       355693
MAGE_REPFLG    352380
IMP_SEX        352316
MRACEIMP       302416
OBGEST_FLG     290891
MHISPX          90000
RF_CESAR        52691
RF_CESARN       52691
dtype: int64

 Saved clinical_training_data.csv — ready for train_clinical_model.py


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv")
print("Shape:", df.shape)
print("Class balance:\n", df["OUTCOME"].value_counts(normalize=True))
print("\nSample rows:\n", df.sample(5))

Shape: (718136, 80)
Class balance:
 OUTCOME
0    0.501243
1    0.498757
Name: proportion, dtype: float64

Sample rows:
         BFACIL  BFACIL3 MAGE_IMPFLG MAGE_REPFLG  MAGER  MAGER14  MAGER9  \
694147     1.0        1         NaN         NaN     31       10       5   
217633     1.0        1         NaN         NaN     43       12       7   
246335     1.0        1                             30       10       5   
409123     1.0        1                             33       10       5   
576641     1.0        1                             25        9       4   

        MBSTATE_REC  RESTATUS  MRACE6  ...  F_RF_CESAR F_RF_NCESAR  F_MM_AICU  \
694147            1         2     2.0  ...         1.0         1.0        1.0   
217633            1         2     1.0  ...         1.0         1.0        1.0   
246335            1         2     1.0  ...         1.0         1.0        1.0   
409123            1         1     2.0  ...         1.0         1.0        1.0   
576641            1     

In [ ]:
# Paste this whole script into a new Colab cell — it already has the
# correct Drive path built in, ready to run as-is

DATA_PATH = "/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv"
MODEL_OUT_PATH = "/content/drive/MyDrive/fetal-health-project/models/stream3_clinical_xgb.json"

# --- paste the rest of the script's contents below this line ---

In [ ]:
#Stram - 3 value check and AUC check from tabular dataset

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import os

RANDOM_SEED = 42
DATA_PATH = "/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv"
MODEL_OUT_PATH = "/content/drive/MyDrive/fetal-health-project/models/stream3_clinical_xgb.json"

TARGET_COLUMN = "OUTCOME"

# Retrain WITHOUT COMBGEST to get an honest "floor" performance number
# using only genuine maternal risk factors, no registry-timing artifact

NUMERIC_FEATURES_NO_GA = ["MAGER", "BMI", "PRECARE", "PRIORDEAD", "PRIORLIVE", "MEDUC"]

X_no_ga = df[NUMERIC_FEATURES_NO_GA + CATEGORICAL_FEATURES]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X_no_ga, y, test_size=0.2, random_state=42, stratify=y)

# reuse your existing build_pipeline(), just swap which NUMERIC_FEATURES list it references



CATEGORICAL_FEATURES = [
    "RF_GDIAB", "RF_GHYPE", "RF_EHYPE", "RF_CESAR", "RF_ARTEC", "RF_INFTR",
    "RF_FEDRG", "CIG_REC", "WIC", "DPLURAL", "SEX", "MRACE6",
    "MBSTATE_REC",
]

UNKNOWN_CODES = ["U", "9", "99", "Unknown"]
NUMERIC_UNKNOWN_CODES = {"COMBGEST": [99]}

def load_data(path=DATA_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}.")
    df = pd.read_csv(path)
    print(f"Loaded combined clinical dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    missing_needed = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]
                       if c not in df.columns]
    if missing_needed:
        raise KeyError(f"Expected column(s) not found: {missing_needed}")
    return df

def clean_unknown_codes(df):
    before_counts = {col: (df[col].isin(UNKNOWN_CODES)).sum() for col in CATEGORICAL_FEATURES}
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].replace(UNKNOWN_CODES, np.nan)
    print(f"Replaced {sum(before_counts.values())} Unknown-coded categorical values")
    for col, codes in NUMERIC_UNKNOWN_CODES.items():
        before = df[col].isin(codes).sum()
        df[col] = df[col].replace(codes, np.nan)
        print(f"{col}: {before} sentinel values {codes} converted to NaN")
    return df

def build_pipeline():
    preprocessor = ColumnTransformer(transformers=[
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("categorical", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_FEATURES),
    ])
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=RANDOM_SEED,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])

def train(df):
    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = df[TARGET_COLUMN]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]
    print("\n--- Stream 3 Clinical Risk Model: Evaluation ---")
    print(classification_report(y_test, preds, target_names=["Healthy/Live Birth", "Stillbirth"]))
    print(f"AUC-ROC: {roc_auc_score(y_test, proba):.4f}")
    return pipeline

df = load_data()
df = clean_unknown_codes(df)

print("\nCOMBGEST by outcome, after cleaning:")
print(df.groupby("OUTCOME")["COMBGEST"].describe())

pipeline = train(df)

os.makedirs(os.path.dirname(MODEL_OUT_PATH), exist_ok=True)
pipeline.named_steps["model"].save_model(MODEL_OUT_PATH)
print(f"\nModel saved to {MODEL_OUT_PATH}")

Loaded combined clinical dataset: 718136 rows, 80 columns
Replaced 1039563 Unknown-coded categorical values
COMBGEST: 12188 sentinel values [99] converted to NaN

COMBGEST by outcome, after cleaning:
            count       mean        std   min   25%   50%   75%   max
OUTCOME                                                              
0        359685.0  38.559067   2.475167  17.0  38.0  39.0  40.0  47.0
1        346263.0  19.604070  10.764422   2.0  10.0  20.0  28.0  47.0

--- Stream 3 Clinical Risk Model: Evaluation ---
                    precision    recall  f1-score   support

Healthy/Live Birth       0.90      0.95      0.92     71993
        Stillbirth       0.95      0.89      0.92     71635

          accuracy                           0.92    143628
         macro avg       0.92      0.92      0.92    143628
      weighted avg       0.92      0.92      0.92    143628

AUC-ROC: 0.9684

Model saved to /content/drive/MyDrive/fetal-health-project/models/stream3_clinical_xgb.jso

In [ ]:
feature_names = (NUMERIC_FEATURES +
    list(pipeline.named_steps["preprocess"].transformers_[1][1]
         .named_steps["onehot"].get_feature_names_out(CATEGORICAL_FEATURES)))

importances = pipeline.named_steps["model"].feature_importances_
top_15 = sorted(zip(feature_names, importances), key=lambda x: -x[1])[:15]

for name, score in top_15:
    print(f"{name}: {score:.4f}")

COMBGEST: 0.4336
DMETH_REC_9: 0.1361
PRIORDEAD: 0.0890
PRECARE: 0.0872
MBSTATE_REC_3: 0.0393
DMETH_REC_2: 0.0367
MEDUC: 0.0287
WIC_N: 0.0155
DMETH_REC_1: 0.0119
WIC_Y: 0.0112
DPLURAL_1: 0.0109
CIG_REC_Y: 0.0079
MRACE6_2.0: 0.0075
BMI: 0.0068
CIG_REC_N: 0.0062


In [ ]:
print(df[df["OUTCOME"] == 1]["COMBGEST"].value_counts().sort_index().head(20))

COMBGEST
2.0       962
3.0      1442
4.0      3410
5.0      7115
6.0     14921
7.0     13973
8.0     21057
9.0     20148
10.0    19002
11.0    14487
12.0    11676
13.0     7840
14.0     5939
15.0     5475
16.0     6216
17.0     6211
18.0     6298
19.0     6926
20.0    16564
21.0    15884
Name: count, dtype: int64


In [ ]:
#Layer - 3 , Hadlock Formula applying


def calculate_efw(hc_mm, bpd_mm, ac_mm, fl_mm):
    """Hadlock 1985 four-parameter EFW formula.
    Inputs in millimeters (matching your pipeline's standardized units) —
    internally converted to cm, since Hadlock's coefficients are cm-calibrated."""
    hc_cm, bpd_cm, ac_cm, fl_cm = hc_mm/10, bpd_mm/10, ac_mm/10, fl_mm/10
    log10_efw = (
        1.3596
        - 0.00386 * (ac_cm * fl_cm)
        + 0.0064 * hc_cm
        + 0.00061 * (bpd_cm * ac_cm)
        + 0.0424 * ac_cm
        + 0.174 * fl_cm
    )
    return 10 ** log10_efw

efw = calculate_efw(hc_mm=280, bpd_mm=80, ac_mm=250, fl_mm=55)
print(f"Test EFW: {efw:.1f} grams")  # should print ~1403.0

Test EFW: 1403.0 grams


In [ ]:
import math
from scipy.stats import norm

def calculate_efw(hc_mm, bpd_mm, ac_mm, fl_mm):
    """Hadlock 1985 four-parameter EFW formula. Inputs in mm, converted
    to cm internally since Hadlock's coefficients are cm-calibrated."""
    hc_cm, bpd_cm, ac_cm, fl_cm = hc_mm/10, bpd_mm/10, ac_mm/10, fl_mm/10
    log10_efw = (1.3596 - 0.00386*(ac_cm*fl_cm) + 0.0064*hc_cm
                 + 0.00061*(bpd_cm*ac_cm) + 0.0424*ac_cm + 0.174*fl_cm)
    return 10 ** log10_efw

def hadlock_reference_mean_sd(ga_weeks):
    """Hadlock et al. 1991 fetal weight reference standard.
    Mean EFW from the paper's published regression equation;
    SD as a uniform 12.7% coefficient of variation, per the paper."""
    mean = math.exp(0.578 + 0.332*ga_weeks - 0.00354*(ga_weeks**2))
    sd = 0.127 * mean
    return mean, sd

def growth_percentile(efw_grams, ga_weeks):
    """Converts an EFW at a given gestational age into a percentile
    against the Hadlock 1991 reference standard."""
    mean, sd = hadlock_reference_mean_sd(ga_weeks)
    z = (efw_grams - mean) / sd
    percentile = norm.cdf(z) * 100
    return percentile, z

def flag_growth_restriction(percentile, threshold=10):
    """Standard clinical convention: <10th percentile = growth-restricted."""
    return percentile < threshold

In [ ]:
# HC18 datset check

from torchvision import transforms

# Data augmentation for training (matches your project's original plan)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Validation data — NO augmentation, we want to evaluate on clean images
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def train_hc_model(train_loader, val_loader, max_epochs=150, lr=1e-4, patience=10):
    model = build_hc_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                preds = model(images)
                val_loss += criterion(preds, targets).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{max_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} "
              f"- lr: {optimizer.param_groups[0]['lr']:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\nNo improvement for {patience} epochs — stopping early at epoch {epoch+1}.")
            break

    model.load_state_dict(best_model_state)  # restore the BEST version, not the last
    print(f"\nBest val_loss achieved: {best_val_loss:.4f} (RMSE ~ {best_val_loss**0.5:.1f} mm combined)")
    return model

In [ ]:
print(DRIVE_ROOT, DEVICE, HC18Dataset, build_hc_model)

/content/drive/MyDrive/fetal-health-project/data/raw/HC18 cuda <class '__main__.HC18Dataset'> <function build_hc_model at 0x7f08ab434860>


In [ ]:
# ---- Setup ----
base_dataset = HC18Dataset(
    csv_path=f"{DRIVE_ROOT}/training_set_pixel_size_and_HC.csv",
    image_dir=f"{DRIVE_ROOT}/training_set",
    transform=val_transform
)

val_size = int(0.15 * len(base_dataset))
train_size = len(base_dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(base_dataset, [train_size, val_size])

train_ds.dataset = HC18Dataset(
    csv_path=f"{DRIVE_ROOT}/training_set_pixel_size_and_HC.csv",
    image_dir=f"{DRIVE_ROOT}/training_set",
    transform=train_transform
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

print(f"Training on {DEVICE} | train: {len(train_ds)}, val: {len(val_ds)}")

model = train_hc_model(train_loader, val_loader, max_epochs=150, patience=10)

os.makedirs("/content/drive/MyDrive/fetal-health-project/models", exist_ok=True)
torch.save(model.state_dict(), "/content/drive/MyDrive/fetal-health-project/models/hc18_convnext_tiny.pt")
print("Model saved.")

Training on cuda | train: 850, val: 149
Epoch 1/150 - train_loss: 12395.3704 - val_loss: 10617.3188 - lr: 0.000100
Epoch 2/150 - train_loss: 10681.6987 - val_loss: 9319.9950 - lr: 0.000100
Epoch 3/150 - train_loss: 9385.2678 - val_loss: 8089.3546 - lr: 0.000100
Epoch 4/150 - train_loss: 8181.9995 - val_loss: 7006.7763 - lr: 0.000100
Epoch 5/150 - train_loss: 7144.4819 - val_loss: 6077.0413 - lr: 0.000100
Epoch 6/150 - train_loss: 6402.2310 - val_loss: 5255.1259 - lr: 0.000100
Epoch 7/150 - train_loss: 5549.2342 - val_loss: 4562.5183 - lr: 0.000100
Epoch 8/150 - train_loss: 4896.5661 - val_loss: 3995.3847 - lr: 0.000100
Epoch 9/150 - train_loss: 4335.9133 - val_loss: 3500.3112 - lr: 0.000100
Epoch 10/150 - train_loss: 3934.0757 - val_loss: 3080.3417 - lr: 0.000100
Epoch 11/150 - train_loss: 3510.8605 - val_loss: 2826.7500 - lr: 0.000100
Epoch 12/150 - train_loss: 3329.9629 - val_loss: 2410.5331 - lr: 0.000100
Epoch 13/150 - train_loss: 2741.9098 - val_loss: 2295.6549 - lr: 0.000100
Epoc

In [ ]:
#checking fetus HC and BPD ratio value where they are perfectly initiallized or not



import pandas as pd

df = pd.read_csv(f"{DRIVE_ROOT}/training_set_pixel_size_and_HC.csv")

ratios = []
for _, row in df.sample(20, random_state=42).iterrows():
    mask_filename = row["filename"].replace(".png", "_Annotation.png")
    mask_path = f"{DRIVE_ROOT}/training_set/{mask_filename}"
    bpd = extract_bpd_from_mask(mask_path, row["pixel size(mm)"])
    hc_given = row["head circumference (mm)"]
    ratio = hc_given / bpd
    ratios.append(ratio)
    print(f"{row['filename']}: HC={hc_given:.1f}mm, BPD={bpd:.1f}mm, ratio={ratio:.2f}")

import numpy as np
print(f"\nAverage HC/BPD ratio across sample: {np.mean(ratios):.2f}")
print("(Expect roughly 2.8-3.5 for a normal head shape — confirms extraction is anatomically correct)")

368_HC.png: HC=166.2mm, BPD=42.6mm, ratio=3.90
643_HC.png: HC=213.6mm, BPD=60.4mm, ratio=3.54
166_HC.png: HC=116.8mm, BPD=34.5mm, ratio=3.39
245_2HC.png: HC=159.5mm, BPD=41.6mm, ratio=3.83
599_HC.png: HC=186.9mm, BPD=52.3mm, ratio=3.57
469_2HC.png: HC=166.7mm, BPD=49.4mm, ratio=3.37
725_HC.png: HC=287.1mm, BPD=81.7mm, ratio=3.51
441_HC.png: HC=176.5mm, BPD=47.5mm, ratio=3.71
355_HC.png: HC=180.6mm, BPD=51.0mm, ratio=3.54
549_HC.png: HC=169.5mm, BPD=47.1mm, ratio=3.60
756_HC.png: HC=283.0mm, BPD=80.6mm, ratio=3.51
513_HC.png: HC=178.5mm, BPD=50.2mm, ratio=3.56
704_HC.png: HC=269.0mm, BPD=74.4mm, ratio=3.62
487_HC.png: HC=176.5mm, BPD=50.9mm, ratio=3.47
583_2HC.png: HC=183.0mm, BPD=50.1mm, ratio=3.65
059_HC.png: HC=76.2mm, BPD=21.9mm, ratio=3.48
671_2HC.png: HC=244.8mm, BPD=70.3mm, ratio=3.48
381_HC.png: HC=170.4mm, BPD=46.0mm, ratio=3.71
308_HC.png: HC=166.9mm, BPD=47.5mm, ratio=3.52
165_HC.png: HC=112.7mm, BPD=33.2mm, ratio=3.39

Average HC/BPD ratio across sample: 3.57
(Expect roughly

In [ ]:
#value check of HC MAE error , if abnormal value are not found here then have to procced next


model.eval()
hc_errors, bpd_errors = [], []

with torch.no_grad():
    for images, targets in val_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        preds = model(images)
        hc_errors.extend((preds[:, 0] - targets[:, 0]).abs().cpu().tolist())
        bpd_errors.extend((preds[:, 1] - targets[:, 1]).abs().cpu().tolist())

import numpy as np
hc_mae = np.mean(hc_errors)
bpd_mae = np.mean(bpd_errors)
hc_rmse = np.sqrt(np.mean(np.array(hc_errors)**2))
bpd_rmse = np.sqrt(np.mean(np.array(bpd_errors)**2))

print(f"HC  -> MAE: {hc_mae:.2f}mm, RMSE: {hc_rmse:.2f}mm")
print(f"BPD -> MAE: {bpd_mae:.2f}mm, RMSE: {bpd_rmse:.2f}mm")

HC  -> MAE: 10.22mm, RMSE: 16.07mm
BPD -> MAE: 3.03mm, RMSE: 4.75mm


In [ ]:
### HC mainly still came out with the worse relative error.


#dominant on combined error. 10x then BPD cuz When two targets sit on very different scales in the same loss function,
#the optimizer effectively gets pulled around unevenly, which can make training noisier for both targets rather than
#cleanly prioritizing the bigger one.
#putting both HC and BPD on the same scale (mean 0, std 1) before computing the loss,
#so the model receives balanced, comparable error signals for both, then converting back to real millimeters



import numpy as np

# ---- Step 1: compute normalization stats from the TRAINING split only ----
# (never include validation data when computing these — that would leak
# information about val data into training, a subtle but real mistake to avoid)

train_indices = train_ds.indices
train_csv = base_dataset.data.iloc[train_indices]

hc_values = train_csv["head circumference (mm)"].values
hc_mean, hc_std = hc_values.mean(), hc_values.std()

bpd_values = []
for _, row in train_csv.iterrows():
    mask_filename = row["filename"].replace(".png", "_Annotation.png")
    mask_path = f"{DRIVE_ROOT}/training_set/{mask_filename}"
    bpd_values.append(extract_bpd_from_mask(mask_path, row["pixel size(mm)"]))
bpd_values = np.array(bpd_values)
bpd_mean, bpd_std = bpd_values.mean(), bpd_values.std()

print(f"HC  stats -> mean: {hc_mean:.1f}, std: {hc_std:.1f}")
print(f"BPD stats -> mean: {bpd_mean:.1f}, std: {bpd_std:.1f}")

target_mean = torch.tensor([hc_mean, bpd_mean], dtype=torch.float32).to(DEVICE)
target_std = torch.tensor([hc_std, bpd_std], dtype=torch.float32).to(DEVICE)


def train_hc_model_normalized(train_loader, val_loader, target_mean, target_std,
                               max_epochs=150, lr=1e-4, patience=10):
    model = build_hc_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            targets_norm = (targets - target_mean) / target_std  # <-- normalize here

            optimizer.zero_grad()
            preds_norm = model(images)
            loss = criterion(preds_norm, targets_norm)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                targets_norm = (targets - target_mean) / target_std
                preds_norm = model(images)
                val_loss += criterion(preds_norm, targets_norm).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{max_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} "
              f"- lr: {optimizer.param_groups[0]['lr']:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\nNo improvement for {patience} epochs — stopping early at epoch {epoch+1}.")
            break

    model.load_state_dict(best_model_state)
    return model


# ---- Train with normalization ----
model = train_hc_model_normalized(train_loader, val_loader, target_mean, target_std,
                                   max_epochs=150, patience=10)

os.makedirs("/content/drive/MyDrive/fetal-health-project/models", exist_ok=True)
torch.save(model.state_dict(), "/content/drive/MyDrive/fetal-health-project/models/hc18_convnext_tiny_normalized.pt")
print("Model saved.")

# ---- Evaluate in REAL mm — must denormalize predictions first ----
model.eval()
hc_errors, bpd_errors = [], []
with torch.no_grad():
    for images, targets in val_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        preds_norm = model(images)
        preds = preds_norm * target_std + target_mean  # <-- denormalize back to real mm
        hc_errors.extend((preds[:, 0] - targets[:, 0]).abs().cpu().tolist())
        bpd_errors.extend((preds[:, 1] - targets[:, 1]).abs().cpu().tolist())

hc_mae = np.mean(hc_errors)
bpd_mae = np.mean(bpd_errors)
hc_rmse = np.sqrt(np.mean(np.array(hc_errors)**2))
bpd_rmse = np.sqrt(np.mean(np.array(bpd_errors)**2))

print(f"\nHC  -> MAE: {hc_mae:.2f}mm, RMSE: {hc_rmse:.2f}mm")
print(f"BPD -> MAE: {bpd_mae:.2f}mm, RMSE: {bpd_rmse:.2f}mm")

HC  stats -> mean: 174.3, std: 66.3
BPD stats -> mean: 49.0, std: 18.6
Epoch 1/150 - train_loss: 0.7104 - val_loss: 0.2788 - lr: 0.000100
Epoch 2/150 - train_loss: 0.1620 - val_loss: 0.1989 - lr: 0.000100
Epoch 3/150 - train_loss: 0.0894 - val_loss: 0.1991 - lr: 0.000100
Epoch 4/150 - train_loss: 0.0827 - val_loss: 0.0767 - lr: 0.000100
Epoch 5/150 - train_loss: 0.0499 - val_loss: 0.0762 - lr: 0.000100
Epoch 6/150 - train_loss: 0.0363 - val_loss: 0.1058 - lr: 0.000100
Epoch 7/150 - train_loss: 0.0387 - val_loss: 0.0962 - lr: 0.000100
Epoch 8/150 - train_loss: 0.0381 - val_loss: 0.2326 - lr: 0.000100
Epoch 9/150 - train_loss: 0.0553 - val_loss: 0.0709 - lr: 0.000100
Epoch 10/150 - train_loss: 0.0229 - val_loss: 0.0671 - lr: 0.000100
Epoch 11/150 - train_loss: 0.0238 - val_loss: 0.0693 - lr: 0.000100
Epoch 12/150 - train_loss: 0.0293 - val_loss: 0.0754 - lr: 0.000100
Epoch 13/150 - train_loss: 0.0202 - val_loss: 0.0731 - lr: 0.000100
Epoch 14/150 - train_loss: 0.0174 - val_loss: 0.0610 -

In [1]:
#Apporach 1 : using Huber loss fucntion ;
# which manupulate MAE to MSE base in outliers ot dataset . could be help on loss function gap 10.22 mm to 2-3 mm


import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

DRIVE_ROOT = "/content/drive/MyDrive/fetal-health-project/data/raw/HC18"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def extract_full_ellipse_from_mask(mask_path, pixel_size_mm):
    """Extracts all 5 ellipse geometry parameters — the modern approach
    regresses THIS, not the final HC number directly."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest = max(contours, key=cv2.contourArea)
    (cx, cy), (minor_px, major_px), angle = cv2.fitEllipse(largest)
    semi_major_mm = (major_px / 2) * pixel_size_mm
    semi_minor_mm = (minor_px / 2) * pixel_size_mm
    cx_mm, cy_mm = cx * pixel_size_mm, cy * pixel_size_mm
    return cx_mm, cy_mm, semi_major_mm, semi_minor_mm, angle


def hc_bpd_from_ellipse_params(semi_major_mm, semi_minor_mm):
    """Ramanujan approximation — same formula real sonographers' software uses
    once an ellipse has been fitted."""
    a, b = semi_major_mm, semi_minor_mm
    hc = np.pi * (3*(a+b) - np.sqrt((3*a+b)*(a+3*b)))
    bpd = 2 * min(a, b)
    return hc, bpd


class HC18GeometryDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform=None):
        self.data = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        pixel_size = row["pixel size(mm)"]
        mask_filename = row["filename"].replace(".png", "_Annotation.png")
        mask_path = os.path.join(self.image_dir, mask_filename)
        cx, cy, semi_major, semi_minor, angle = extract_full_ellipse_from_mask(mask_path, pixel_size)

        targets = torch.tensor([cx, cy, semi_major, semi_minor, angle], dtype=torch.float32)
        return image, targets


def build_geometry_model():
    return timm.create_model("convnext_tiny", pretrained=True, num_classes=5)


def train_geometry_model(train_loader, val_loader, target_mean, target_std,
                          max_epochs=150, lr=1e-4, patience=10):
    model = build_geometry_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    criterion = nn.SmoothL1Loss()  # Huber loss — robust to outliers, matches literature

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            targets_norm = (targets - target_mean) / target_std
            optimizer.zero_grad()
            preds_norm = model(images)
            loss = criterion(preds_norm, targets_norm)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                targets_norm = (targets - target_mean) / target_std
                preds_norm = model(images)
                val_loss += criterion(preds_norm, targets_norm).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{max_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} "
              f"- lr: {optimizer.param_groups[0]['lr']:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nStopping early at epoch {epoch+1}.")
            break

    model.load_state_dict(best_model_state)
    return model


# ---- Setup ----
train_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_geo_dataset = HC18GeometryDataset(
    csv_path=f"{DRIVE_ROOT}/training_set_pixel_size_and_HC.csv",
    image_dir=f"{DRIVE_ROOT}/training_set", transform=val_transform
)
val_size = int(0.15 * len(base_geo_dataset))
train_size = len(base_geo_dataset) - val_size
train_ds_geo, val_ds_geo = torch.utils.data.random_split(base_geo_dataset, [train_size, val_size])
train_ds_geo.dataset = HC18GeometryDataset(
    csv_path=f"{DRIVE_ROOT}/training_set_pixel_size_and_HC.csv",
    image_dir=f"{DRIVE_ROOT}/training_set", transform=train_transform
)

# Compute normalization stats from training split only
train_indices = train_ds_geo.indices
all_targets = torch.stack([base_geo_dataset[i][1] for i in train_indices])
target_mean = all_targets.mean(dim=0).to(DEVICE)
target_std = all_targets.std(dim=0).to(DEVICE)
print("Target mean:", target_mean.cpu().tolist())
print("Target std:", target_std.cpu().tolist())

train_loader_geo = DataLoader(train_ds_geo, batch_size=16, shuffle=True, num_workers=2)
val_loader_geo = DataLoader(val_ds_geo, batch_size=16, shuffle=False, num_workers=2)

print(f"Training geometry model on {DEVICE} | train: {len(train_ds_geo)}, val: {len(val_ds_geo)}")
geo_model = train_geometry_model(train_loader_geo, val_loader_geo, target_mean, target_std, max_epochs=150, patience=10)

# ---- Evaluate in real HC/BPD mm (derived from predicted ellipse) ----
geo_model.eval()
hc_errors, bpd_errors = [], []
with torch.no_grad():
    for images, targets in val_loader_geo:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        preds_norm = geo_model(images)
        preds = preds_norm * target_std + target_mean
        for i in range(len(preds)):
            pred_hc, pred_bpd = hc_bpd_from_ellipse_params(preds[i, 2].item(), preds[i, 3].item())
            true_hc, true_bpd = hc_bpd_from_ellipse_params(targets[i, 2].item(), targets[i, 3].item())
            hc_errors.append(abs(pred_hc - true_hc))
            bpd_errors.append(abs(pred_bpd - true_bpd))

print(f"\n[GEOMETRY APPROACH] HC  -> MAE: {np.mean(hc_errors):.2f}mm, RMSE: {np.sqrt(np.mean(np.array(hc_errors)**2)):.2f}mm")
print(f"[GEOMETRY APPROACH] BPD -> MAE: {np.mean(bpd_errors):.2f}mm, RMSE: {np.sqrt(np.mean(np.array(bpd_errors)**2)):.2f}mm")

os.makedirs("/content/drive/MyDrive/fetal-health-project/models", exist_ok=True)
torch.save(geo_model.state_dict(), "/content/drive/MyDrive/fetal-health-project/models/hc18_geometry_convnext.pt")
print("Geometry model saved.")

Target mean: [54.84204864501953, 37.71098709106445, 31.065813064575195, 24.555099487304688, 89.66671752929688]
Target std: [21.032228469848633, 14.67557144165039, 11.468890190124512, 9.023344039916992, 22.760433197021484]
Training geometry model on cuda | train: 850, val: 149


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1/150 - train_loss: 0.3100 - val_loss: 0.3939 - lr: 0.000100
Epoch 2/150 - train_loss: 0.1560 - val_loss: 0.1753 - lr: 0.000100
Epoch 3/150 - train_loss: 0.1154 - val_loss: 0.1837 - lr: 0.000100
Epoch 4/150 - train_loss: 0.1158 - val_loss: 0.1571 - lr: 0.000100
Epoch 5/150 - train_loss: 0.1008 - val_loss: 0.1529 - lr: 0.000100
Epoch 6/150 - train_loss: 0.0901 - val_loss: 0.1348 - lr: 0.000100
Epoch 7/150 - train_loss: 0.0837 - val_loss: 0.1536 - lr: 0.000100
Epoch 8/150 - train_loss: 0.0786 - val_loss: 0.1369 - lr: 0.000100
Epoch 9/150 - train_loss: 0.0731 - val_loss: 0.1588 - lr: 0.000100
Epoch 10/150 - train_loss: 0.0685 - val_loss: 0.1446 - lr: 0.000100
Epoch 11/150 - train_loss: 0.0642 - val_loss: 0.1419 - lr: 0.000100
Epoch 12/150 - train_loss: 0.0657 - val_loss: 0.1541 - lr: 0.000050
Epoch 13/150 - train_loss: 0.0401 - val_loss: 0.1524 - lr: 0.000050
Epoch 14/150 - train_loss: 0.0311 - val_loss: 0.1332 - lr: 0.000050
Epoch 15/150 - train_loss: 0.0259 - val_loss: 0.1269 - lr

In [3]:
!pip install segmentation-models-pytorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


In [5]:
#Approach C — Multi-task U-Net (segmentation + ellipse regression, shared encoder)


import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

DRIVE_ROOT = "/content/drive/MyDrive/fetal-health-project/data/raw/HC18"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def extract_full_ellipse_from_mask(mask_path, pixel_size_mm):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest = max(contours, key=cv2.contourArea)
    (cx, cy), (minor_px, major_px), angle = cv2.fitEllipse(largest)
    semi_major_mm = (major_px / 2) * pixel_size_mm
    semi_minor_mm = (minor_px / 2) * pixel_size_mm
    return cx * pixel_size_mm, cy * pixel_size_mm, semi_major_mm, semi_minor_mm, angle


def hc_bpd_from_ellipse_params(semi_major_mm, semi_minor_mm):
    a, b = semi_major_mm, semi_minor_mm
    hc = np.pi * (3*(a+b) - np.sqrt((3*a+b)*(a+3*b)))
    bpd = 2 * min(a, b)
    return hc, bpd


class HC18MultiTaskDataset(Dataset):
    """Returns image, binary segmentation mask, AND ellipse geometry — all three,
    since the multi-task model trains on both signals simultaneously."""
    def __init__(self, csv_path, image_dir, transform=None, mask_size=224):
        self.data = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        self.mask_size = mask_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        pixel_size = row["pixel size(mm)"]
        mask_filename = row["filename"].replace(".png", "_Annotation.png")
        mask_path = os.path.join(self.image_dir, mask_filename)

        # Ellipse geometry target
        cx, cy, semi_major, semi_minor, angle = extract_full_ellipse_from_mask(mask_path, pixel_size)
        geo_target = torch.tensor([cx, cy, semi_major, semi_minor, angle], dtype=torch.float32)

        # Segmentation mask target — filled ellipse (not just the thin outline),
        # since Dice/BCE need a solid region to compare against
        raw_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        contours, _ = cv2.findContours(raw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        largest = max(contours, key=cv2.contourArea)
        filled_mask = np.zeros_like(raw_mask)
        cv2.drawContours(filled_mask, [largest], -1, 255, thickness=cv2.FILLED)
        filled_mask = cv2.resize(filled_mask, (self.mask_size, self.mask_size))
        mask_tensor = torch.tensor(filled_mask / 255.0, dtype=torch.float32).unsqueeze(0)

        if self.transform:
            image = self.transform(image)

        return image, mask_tensor, geo_target


def build_multitask_model():
    model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1)
    regression_head = nn.Linear(512, 5)  # resnet34's deepest feature = 512 channels
    return model, regression_head


def train_multitask_model(train_loader, val_loader, target_mean, target_std,
                           max_epochs=150, lr=1e-4, patience=10, reg_weight=0.5):
    model, regression_head = build_multitask_model()
    model, regression_head = model.to(DEVICE), regression_head.to(DEVICE)

    params = list(model.parameters()) + list(regression_head.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    dice_loss_fn = smp.losses.DiceLoss(mode="binary")
    bce_loss_fn = nn.BCEWithLogitsLoss()
    huber_loss_fn = nn.SmoothL1Loss()

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_states = None

    for epoch in range(max_epochs):
        model.train(); regression_head.train()
        train_loss = 0
        for images, masks, geo_targets in train_loader:
            images, masks, geo_targets = images.to(DEVICE), masks.to(DEVICE), geo_targets.to(DEVICE)
            geo_targets_norm = (geo_targets - target_mean) / target_std

            optimizer.zero_grad()
            seg_pred = model(images)
            features = model.encoder(images)[-1]
            pooled = nn.functional.adaptive_avg_pool2d(features, 1).flatten(1)
            geo_pred = regression_head(pooled)

            seg_loss = dice_loss_fn(seg_pred, masks) + bce_loss_fn(seg_pred, masks)
            reg_loss = huber_loss_fn(geo_pred, geo_targets_norm)
            loss = seg_loss + reg_weight * reg_loss

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval(); regression_head.eval()
        val_loss = 0
        with torch.no_grad():
            for images, masks, geo_targets in val_loader:
                images, masks, geo_targets = images.to(DEVICE), masks.to(DEVICE), geo_targets.to(DEVICE)
                geo_targets_norm = (geo_targets - target_mean) / target_std
                seg_pred = model(images)
                features = model.encoder(images)[-1]
                pooled = nn.functional.adaptive_avg_pool2d(features, 1).flatten(1)
                geo_pred = regression_head(pooled)
                seg_loss = dice_loss_fn(seg_pred, masks) + bce_loss_fn(seg_pred, masks)
                reg_loss = huber_loss_fn(geo_pred, geo_targets_norm)
                val_loss += (seg_loss + reg_weight * reg_loss).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{max_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} "
              f"- lr: {optimizer.param_groups[0]['lr']:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_states = (model.state_dict(), regression_head.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nStopping early at epoch {epoch+1}.")
            break

    model.load_state_dict(best_states[0])
    regression_head.load_state_dict(best_states[1])
    return model, regression_head


# ---- Setup ----
transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_mt_dataset = HC18MultiTaskDataset(
    csv_path=f"{DRIVE_ROOT}/training_set_pixel_size_and_HC.csv",
    image_dir=f"{DRIVE_ROOT}/training_set", transform=transform
)
val_size = int(0.15 * len(base_mt_dataset))
train_size = len(base_mt_dataset) - val_size
train_ds_mt, val_ds_mt = torch.utils.data.random_split(base_mt_dataset, [train_size, val_size])

train_indices = train_ds_mt.indices
all_geo_targets = torch.stack([base_mt_dataset[i][2] for i in train_indices])
target_mean = all_geo_targets.mean(dim=0).to(DEVICE)
target_std = all_geo_targets.std(dim=0).to(DEVICE)

train_loader_mt = DataLoader(train_ds_mt, batch_size=16, shuffle=True, num_workers=2)
val_loader_mt = DataLoader(val_ds_mt, batch_size=16, shuffle=False, num_workers=2)

print(f"Training multi-task model on {DEVICE} | train: {len(train_ds_mt)}, val: {len(val_ds_mt)}")
mt_model, mt_reg_head = train_multitask_model(train_loader_mt, val_loader_mt, target_mean, target_std,
                                               max_epochs=150, patience=10)

# ---- Evaluate real HC/BPD mm ----
mt_model.eval(); mt_reg_head.eval()
hc_errors, bpd_errors = [], []
with torch.no_grad():
    for images, masks, geo_targets in val_loader_mt:
        images, geo_targets = images.to(DEVICE), geo_targets.to(DEVICE)
        features = mt_model.encoder(images)[-1]
        pooled = nn.functional.adaptive_avg_pool2d(features, 1).flatten(1)
        preds_norm = mt_reg_head(pooled)
        preds = preds_norm * target_std + target_mean
        for i in range(len(preds)):
            pred_hc, pred_bpd = hc_bpd_from_ellipse_params(preds[i, 2].item(), preds[i, 3].item())
            true_hc, true_bpd = hc_bpd_from_ellipse_params(geo_targets[i, 2].item(), geo_targets[i, 3].item())
            hc_errors.append(abs(pred_hc - true_hc))
            bpd_errors.append(abs(pred_bpd - true_bpd))

print(f"\n[MULTI-TASK APPROACH] HC  -> MAE: {np.mean(hc_errors):.2f}mm, RMSE: {np.sqrt(np.mean(np.array(hc_errors)**2)):.2f}mm")
print(f"[MULTI-TASK APPROACH] BPD -> MAE: {np.mean(bpd_errors):.2f}mm, RMSE: {np.sqrt(np.mean(np.array(bpd_errors)**2)):.2f}mm")

os.makedirs("/content/drive/MyDrive/fetal-health-project/models", exist_ok=True)
torch.save({"unet": mt_model.state_dict(), "reg_head": mt_reg_head.state_dict()},
           "/content/drive/MyDrive/fetal-health-project/models/hc18_multitask.pt")
print("Multi-task model saved.")

Training multi-task model on cuda | train: 850, val: 149
Epoch 1/150 - train_loss: 0.8460 - val_loss: 0.5126 - lr: 0.000100
Epoch 2/150 - train_loss: 0.4291 - val_loss: 0.3473 - lr: 0.000100
Epoch 3/150 - train_loss: 0.2985 - val_loss: 0.2731 - lr: 0.000100
Epoch 4/150 - train_loss: 0.2337 - val_loss: 0.2295 - lr: 0.000100
Epoch 5/150 - train_loss: 0.1855 - val_loss: 0.2001 - lr: 0.000100
Epoch 6/150 - train_loss: 0.1639 - val_loss: 0.1781 - lr: 0.000100
Epoch 7/150 - train_loss: 0.1367 - val_loss: 0.1694 - lr: 0.000100
Epoch 8/150 - train_loss: 0.1244 - val_loss: 0.1624 - lr: 0.000100
Epoch 9/150 - train_loss: 0.1169 - val_loss: 0.1533 - lr: 0.000100
Epoch 10/150 - train_loss: 0.1064 - val_loss: 0.1432 - lr: 0.000100
Epoch 11/150 - train_loss: 0.0909 - val_loss: 0.1406 - lr: 0.000100
Epoch 12/150 - train_loss: 0.0819 - val_loss: 0.1336 - lr: 0.000100
Epoch 13/150 - train_loss: 0.0793 - val_loss: 0.1395 - lr: 0.000100
Epoch 14/150 - train_loss: 0.0685 - val_loss: 0.1292 - lr: 0.000100


In [6]:
# Apporach B : Segmentation technic nnUnet style


import cv2
import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
from torchvision import transforms
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def build_segmentation_model():
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1)


def train_segmentation_model(train_loader, val_loader, max_epochs=150, lr=1e-4, patience=10):
    model = build_segmentation_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    dice_loss_fn = smp.losses.DiceLoss(mode="binary")
    bce_loss_fn = nn.BCEWithLogitsLoss()

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        for images, masks, _ in train_loader:  # ignore geo_target — pure segmentation only
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            preds = model(images)
            loss = dice_loss_fn(preds, masks) + bce_loss_fn(preds, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                preds = model(images)
                val_loss += (dice_loss_fn(preds, masks) + bce_loss_fn(preds, masks)).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{max_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} "
              f"- lr: {optimizer.param_groups[0]['lr']:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nStopping early at epoch {epoch+1}.")
            break

    model.load_state_dict(best_model_state)
    return model


def mask_to_hc_bpd(prob_mask, pixel_size_mm, orig_size, mask_size=224):
    """Converts a predicted probability mask into HC/BPD via ellipse fitting,
    exactly mirroring what real segmentation-based papers do."""
    binary = (prob_mask > 0.5).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None, None  # degenerate prediction — no fallback value invented
    largest = max(contours, key=cv2.contourArea)
    if len(largest) < 5:
        return None, None

    (cx, cy), (minor_px, major_px), angle = cv2.fitEllipse(largest)
    scale_factor = orig_size / mask_size  # mask was resized to 224; scale back
    semi_major_mm = (major_px / 2) * scale_factor * pixel_size_mm
    semi_minor_mm = (minor_px / 2) * scale_factor * pixel_size_mm
    a, b = semi_major_mm, semi_minor_mm
    hc = np.pi * (3*(a+b) - np.sqrt((3*a+b)*(a+3*b)))
    bpd = 2 * min(a, b)
    return hc, bpd


# ---- Reuse the same HC18MultiTaskDataset and splits already built ----
print(f"Training segmentation-only model on {DEVICE} | train: {len(train_ds_mt)}, val: {len(val_ds_mt)}")
seg_model = train_segmentation_model(train_loader_mt, val_loader_mt, max_epochs=150, patience=10)

# ---- Evaluate: predicted mask -> ellipse fit -> HC/BPD ----
seg_model.eval()
hc_errors, bpd_errors = [], []
skipped = 0
with torch.no_grad():
    for idx in val_ds_mt.indices:
        row = base_mt_dataset.data.iloc[idx]
        pixel_size = row["pixel size(mm)"]
        img_path = f"{DRIVE_ROOT}/training_set/{row['filename']}"

        from PIL import Image
        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size
        img_tensor = transform(image).unsqueeze(0).to(DEVICE)

        prob = torch.sigmoid(seg_model(img_tensor))[0, 0].cpu().numpy()

        # Ground truth from the real annotation mask, for fair comparison
        true_hc, true_bpd = hc_bpd_from_ellipse_params(
            *extract_full_ellipse_from_mask(
                f"{DRIVE_ROOT}/training_set/{row['filename'].replace('.png','_Annotation.png')}",
                pixel_size)[2:4]
        )

        pred_hc, pred_bpd = mask_to_hc_bpd(prob, pixel_size, orig_w)
        if pred_hc is None:
            skipped += 1
            continue
        hc_errors.append(abs(pred_hc - true_hc))
        bpd_errors.append(abs(pred_bpd - true_bpd))

print(f"\n[SEGMENTATION APPROACH] HC  -> MAE: {np.mean(hc_errors):.2f}mm, RMSE: {np.sqrt(np.mean(np.array(hc_errors)**2)):.2f}mm")
print(f"[SEGMENTATION APPROACH] BPD -> MAE: {np.mean(bpd_errors):.2f}mm, RMSE: {np.sqrt(np.mean(np.array(bpd_errors)**2)):.2f}mm")
print(f"Skipped (degenerate predictions): {skipped}")

os.makedirs("/content/drive/MyDrive/fetal-health-project/models", exist_ok=True)
torch.save(seg_model.state_dict(), "/content/drive/MyDrive/fetal-health-project/models/hc18_segmentation.pt")
print("Segmentation model saved.")

Training segmentation-only model on cuda | train: 850, val: 149
Epoch 1/150 - train_loss: 0.5829 - val_loss: 0.3166 - lr: 0.000100
Epoch 2/150 - train_loss: 0.2563 - val_loss: 0.2118 - lr: 0.000100
Epoch 3/150 - train_loss: 0.1927 - val_loss: 0.1789 - lr: 0.000100
Epoch 4/150 - train_loss: 0.1463 - val_loss: 0.1494 - lr: 0.000100
Epoch 5/150 - train_loss: 0.1249 - val_loss: 0.1394 - lr: 0.000100
Epoch 6/150 - train_loss: 0.1041 - val_loss: 0.1290 - lr: 0.000100
Epoch 7/150 - train_loss: 0.0958 - val_loss: 0.1272 - lr: 0.000100
Epoch 8/150 - train_loss: 0.0857 - val_loss: 0.1129 - lr: 0.000100
Epoch 9/150 - train_loss: 0.0755 - val_loss: 0.1141 - lr: 0.000100
Epoch 10/150 - train_loss: 0.0673 - val_loss: 0.1073 - lr: 0.000100
Epoch 11/150 - train_loss: 0.0687 - val_loss: 0.1470 - lr: 0.000100
Epoch 12/150 - train_loss: 0.0788 - val_loss: 0.1061 - lr: 0.000100
Epoch 13/150 - train_loss: 0.0616 - val_loss: 0.1004 - lr: 0.000100
Epoch 14/150 - train_loss: 0.0531 - val_loss: 0.0997 - lr: 0.